In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles

plt.style.use('bmh') 
colors = ["#3d5a80","#98c1d9","#e0fbfc","#ee6c4d","#293241"]

# Two concentric rings: inner ring is class 0, outer ring is class 1.
# A smooth 2D regression target (fit with MSE) that needs a curved boundary,
# which makes it a much better showcase for second-order convergence than XOR.
X, y = make_circles(n_samples=100, noise=0.05, factor=0.4, random_state=0)
y = y.reshape(-1, 1).astype(float)

In [ ]:
# NN with one hidden layer: h1 = relu(W_h1 @ x + b_h1),  y = W_o @ h1 + b_o
# A sum of ReLUs is piecewise-linear, so it needs several units to bend a ring-
# shaped boundary. 8 keeps the model tiny but expressive enough for the circles.
# (The width only changes a constant here - the gradient/Hessian math is identical.)
layer_width = 8

def init_params(seed=2):
    rng = np.random.default_rng(seed)
    W_h1 = rng.standard_normal((2, layer_width))  # Hidden layer (2 inputs, `layer_width` nodes)
    b_h1 = np.zeros(layer_width)
    W_o  = rng.standard_normal((layer_width, 1))  # Output layer (`layer_width` inputs, 1 output)
    b_o  = np.zeros(1)
    return W_h1, b_h1, W_o, b_o

# Same starting point for both optimizers, so the convergence curves are comparable.
W_h1, b_h1, W_o, b_o = init_params()

In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_gradient(z):
   return (z > 0).astype(float)

def forward_pass(x, W_h1, b_h1, W_o, b_o):
    z1 = np.dot(x, W_h1) + b_h1
    h1 = relu(z1)
    y_hat = np.dot(h1, W_o) + b_o
    return y_hat, h1, z1


def backward_pass(x, y_true, y_hat, h1, z1, W_o):
    # Works for a single sample or a full batch; the bias gradients sum over samples.
    # Output layer gradients
    gradient_y = y_hat - y_true
    dW_o = np.dot(h1.T, gradient_y)
    db_o = gradient_y.sum(axis=0)

    # Hidden layer gradients
    gradient_h1 = np.dot(gradient_y, W_o.T)
    gradient_z_h1 = gradient_h1 * relu_gradient(z1)
    dW_h1 = np.dot(x.T, gradient_z_h1)
    db_h1 = gradient_z_h1.sum(axis=0)

    return dW_h1, db_h1, dW_o, db_o

In [ ]:
# --- Helpers for the second-order step ---
# The Hessian is defined over *all* parameters at once, so we flatten
# W_h1, b_h1, W_o, b_o into a single vector theta and back.

n_params = W_h1.size + b_h1.size + W_o.size + b_o.size  # 2*w + w + w + 1

def flatten(W_h1, b_h1, W_o, b_o):
    return np.concatenate([W_h1.ravel(), b_h1, W_o.ravel(), b_o])

def unflatten(theta):
    w = layer_width
    i = 0
    W_h1 = theta[i : i + 2 * w].reshape(2, w); i += 2 * w
    b_h1 = theta[i : i + w];                   i += w
    W_o  = theta[i : i + w].reshape(w, 1);     i += w
    b_o  = theta[i : i + 1]
    return W_h1, b_h1, W_o, b_o

def output_jacobian(x, h1, z1, W_o):
    # j = d(y_hat)/d(theta): the backward pass seeded with 1 instead of (y_hat - y).
    dW_o  = h1.T                                # (w, 1)
    db_o  = np.array([1.0])                      # (1,)
    grad_z1 = W_o.T * relu_gradient(z1)          # (1, w)
    dW_h1 = np.dot(x.T, grad_z1)                 # (2, w)
    db_h1 = grad_z1[0]                           # (w,)
    return flatten(dW_h1, db_h1, dW_o, db_o)     # (n_params,)

In [ ]:
# --- Full-batch gradient descent (first-order) ---
W_h1, b_h1, W_o, b_o = init_params()

mse_log = []
learning_rate = 0.5
n_samples = X.shape[0]

for epoch in range(100):
    # 1. Forward pass over the whole dataset at once
    y_hat, h1, z1 = forward_pass(X, W_h1, b_h1, W_o, b_o)

    # 2. Backward pass: one gradient summed over all samples
    dW_h1, db_h1, dW_o, db_o = backward_pass(X, y, y_hat, h1, z1, W_o)

    # 3. Take one step along the averaged gradient
    W_o  -= learning_rate * dW_o  / n_samples
    b_o  -= learning_rate * db_o  / n_samples
    W_h1 -= learning_rate * dW_h1 / n_samples
    b_h1 -= learning_rate * db_h1 / n_samples

    mse_log.append(np.mean(np.square(y - y_hat)))


In [ ]:
# --- Full-batch Gauss-Newton, undamped (second-order) ---
# Same full-batch loop, but we also build the Gauss-Newton Hessian H = J.T @ J
# and solve H @ delta = g instead of just stepping along g.
W_h1, b_h1, W_o, b_o = init_params()

mse_log_newton = []
learning_rate = 0.5
n_samples = X.shape[0]

for epoch in range(100):
    # 1. Forward pass over the whole dataset
    y_hat, h1, z1 = forward_pass(X, W_h1, b_h1, W_o, b_o)
    resid = (y_hat - y).ravel()                  # residuals r_i = y_hat_i - y_i

    # 2. Stack each sample's output gradient d(y_hat_i)/d(theta) into the Jacobian J
    J = np.stack([output_jacobian(X[i:i+1], h1[i:i+1], z1[i:i+1], W_o)
                  for i in range(n_samples)])     # (n_samples, n_params)

    g = J.T @ resid                              # gradient        (n_params,)
    H = J.T @ J                                  # Gauss-Newton Hessian (n_params, n_params)

    # 3. Newton step. A ReLU net has redundant parameters (you can rescale a unit's
    #    input and output weights for the same output), so H is rank-deficient and has
    #    no true inverse. lstsq gives the minimum-norm solution of H @ delta = g,
    #    which is the pseudo-inverse step -- no damping constant needed.
    delta = np.linalg.lstsq(H, g, rcond=None)[0]
    theta = flatten(W_h1, b_h1, W_o, b_o) - learning_rate * delta
    W_h1, b_h1, W_o, b_o = unflatten(theta)

    mse_log_newton.append(np.mean(np.square(y - y_hat)))


In [ ]:
# --- Full-batch Gauss-Newton, somewhat damped (Levenberg-Marquardt) ---
# Same as above, but add lam * I to the Hessian. This makes it invertible again
# and pulls the step slightly toward plain gradient descent, which smooths out the
# early overshoot of the undamped version at the cost of a bit of early speed.
W_h1, b_h1, W_o, b_o = init_params()

mse_log_damped = []
learning_rate = 0.5
lam = 10.0            # somewhere between pure Newton (lam->0) and gradient descent (lam->inf)
n_samples = X.shape[0]

for epoch in range(100):
    # 1. Forward pass over the whole dataset
    y_hat, h1, z1 = forward_pass(X, W_h1, b_h1, W_o, b_o)
    resid = (y_hat - y).ravel()

    # 2. Gauss-Newton Hessian + damping
    J = np.stack([output_jacobian(X[i:i+1], h1[i:i+1], z1[i:i+1], W_o)
                  for i in range(n_samples)])
    g = J.T @ resid
    H = J.T @ J + lam * np.eye(n_params)   # damping makes H invertible

    # 3. Newton step (H is now full-rank, so a plain solve works)
    delta = np.linalg.solve(H, g)
    theta = flatten(W_h1, b_h1, W_o, b_o) - learning_rate * delta
    W_h1, b_h1, W_o, b_o = unflatten(theta)

    mse_log_damped.append(np.mean(np.square(y - y_hat)))


In [ ]:
plt.plot(mse_log, c=colors[3], label="Gradient descent (first-order)")
plt.plot(mse_log_damped, c=colors[4], label="Gauss-Newton, damped (λ=10)")
plt.plot(mse_log_newton, c=colors[0], label="Gauss-Newton, undamped")
plt.xlabel('Iteration')
plt.ylabel('MSE')
plt.legend()
plt.savefig("plot.png")